## Prompt Registry with MLflow

### Installing Utilities and Libraries

In [1]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14 langgraph==1.2.10 mlflow==3.15.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Instantiating the ChatAnthropic Class

In [3]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Enable MLflow Tracing

In [4]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("Prompt-Registry-Lab")
mlflow.set_tracking_uri("http://localhost:5001")

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/27 13:07:11 INFO mlflow.tracking.fluent: Experiment with name 'Prompt-Registry-Lab' does not exist. Creating a new experiment.


### Create your First Prompt

In [5]:
prompt_name = "restaurant_review_analyzer"

initial_template = """
You are an AI assistant analyzing restaurant customer reviews.

Analyze the following customer review:

{{review}}

Provide:
1. The overall sentiment
2. The main topic discussed
3. A short response to the customer

Write the response using a {{tone}} tone.
"""

prompt = mlflow.genai.register_prompt(
    name=f"{prompt_name}",
    template=initial_template,
    commit_message="Initial restaurant review analysis prompt",
    tags={
        "use_case": "customer_review_analysis",
        "task": "sentiment_analysis",
        "version_stage": "development"
    }
)

# set a production alias
mlflow.genai.set_prompt_alias(
    name=f"{prompt_name}",
    alias="production",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

2026/09/27 13:08:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: restaurant_review_analyzer, version 1


Prompt: restaurant_review_analyzer
Version: 1


### Create a Review Analysis Function LLM Call

In [6]:
from langchain_core.messages import HumanMessage

@mlflow.trace
def analyze_review(prompt, review: str, tone: str):

    # Fill the Prompt Registry template variables
    formatted_prompt = prompt.format(
        review=review,
        tone=tone
    )

    response = model.invoke(
        [
          HumanMessage(formatted_prompt)
        ]
    )

    return response.content

### Load and Execute Version 1 of the Prompt

In [7]:
# Load and use the prompt in your application
prompt_v1 = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{prompt_name}@production")

review = """
The pasta was fantastic and our waiter was extremely friendly.
However, we had to wait almost 40 minutes for our food.
"""

result = analyze_review(
    prompt = prompt_v1,
    review=review,
    tone="professional"
)

print(result)

# Review Analysis

## 1. Overall Sentiment
**Mixed (Leaning Neutral-Positive)**
The review contains genuine praise alongside a notable complaint, indicating a moderately positive experience tempered by service efficiency concerns.

## 2. Main Topic Discussed
**Food quality and service speed**
- Positive aspects: Food taste (pasta) and staff friendliness (waiter)
- Negative aspect: Wait time for food delivery (~40 minutes)

## 3. Suggested Response to Customer

---

Dear Valued Customer,

Thank you for taking the time to share your feedback with us. We're delighted to hear that you enjoyed our pasta and that our waiter provided you with friendly service—we'll be sure to pass along your kind words to our team.

That said, we sincerely apologize for the extended wait time you experienced for your food. We understand how frustrating this can be, and we take this feedback seriously. We are currently reviewing our kitchen workflow and staffing during peak hours to ensure faster service witho

### Create Version 2 of the Prompt

In [8]:
improved_template = """
You are an expert customer experience analyst.

Analyze the following restaurant review:

{{review}}

Return your analysis using exactly this structure:

Sentiment:
Positive, Negative, or Mixed

Primary Topic:
Identify the primary topic discussed by the customer.

Key Issue:
Identify the most important issue or compliment.

Recommended Action:
Recommend one action the restaurant should take.

Customer Response:
Write a concise response to the customer using a {{tone}} tone.

Do not invent information that is not contained in the review.
"""

Trace(trace_id=tr-d52c7160195fe33c4ca255ad93e25c1a)

In [9]:
prompt_v2 = mlflow.genai.register_prompt(
    name=f"{prompt_name}",
    template=improved_template,
    commit_message="Added structured analysis and recommended action",
    tags={
        "use_case": "customer_review_analysis",
        "task": "sentiment_analysis",
        "version_stage": "improved"
    }
)

# set a production alias
mlflow.genai.set_prompt_alias(
    name=f"{prompt_name}",
    alias="development",
    version=2
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

2026/09/27 13:21:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: restaurant_review_analyzer, version 2


Prompt: restaurant_review_analyzer
Version: 1


### Load and Execute Version 2 of the Prompt

In [10]:
# Load and use the prompt in your application
prompt_v2 = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{prompt_name}@development")

review = """
The pasta was fantastic and our waiter was extremely friendly.
However, we had to wait almost 40 minutes for our food.
"""

result = analyze_review(
    prompt = prompt_v2,
    review=review,
    tone="professional"
)

print(result)

[{'signature': 'EsUCCpABCBIYAipA23+gAVrzG+ThfSwh5z6axPM4Z4wrSmk+26b0+kFCTPhyjuuWc6RW9p5FHzwEpLvWFL4NpHfJDPIXs+q2CZmd1zIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ4YTdhNWVmNC1iYjMxLTQwYTItYTA0My02NjVhYjJkZGU4NDaoAY2W49UGEgyMYEYtatP2IJ2BNZ8aDMZw3t8SqNNqC3AkCCIwZSXyZ6bn9+3+lCdTHbAPwoGm+eZc10M3uq5S4acbscTjgzSzJPvn3W5/g5Ru+nqIKmIb3IP2OiATOfi2V5uX/c1xfZhZPDmiP7LAUOsM6l2DuJSvdmIeyAdmthGcWANBAsZHwkPvw9v1mwgBpino4IoFvw0HqRou1nfGIitkC5KWl/Lvbmy4BulUP/Dp8N5OzHjOjxgB', 'thinking': '', 'type': 'thinking'}, {'text': "Sentiment:\nMixed\n\nPrimary Topic:\nFood quality and service speed\n\nKey Issue:\nThe pasta and waiter's friendliness were praised, but the 40-minute wait time for food was a significant complaint.\n\nRecommended Action:\nReview kitchen workflow and staffing during peak hours to reduce food preparation and delivery times.\n\nCustomer Response:\nThank you for sharing your feedback. We're delighted to hear you enjoyed the pasta and that our waiter provided friendly service. However, we sinc